# Evolución de métricas de representación por época

Este notebook carga el archivo procesado por `parse_out_repr_epochs.py` y grafica cada métrica por época, donde **cada línea corresponde a un modelo (`arch`)**.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

In [ ]:
# Cambia esta ruta si guardaste el output con otro nombre o en otro formato.
DATA_PATH = "cars3d_repr_epochs.pkl"
df = pd.read_pickle(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
# (Opcional) filtra para un subset concreto
DATASET = None        # e.g. 'cars3d'
EXPERIMENT = None     # e.g. 'orthotopic'
SPLIT = None          # e.g. 'composition_0.1'

plot_df = df.copy()
if DATASET is not None:
    plot_df = plot_df[plot_df['dataset'] == DATASET]
if EXPERIMENT is not None:
    plot_df = plot_df[plot_df['experiment'] == EXPERIMENT]
if SPLIT is not None:
    plot_df = plot_df[plot_df['split'] == SPLIT]

print(plot_df.shape)

In [ ]:
id_cols = {'dataset', 'experiment', 'split', 'c', 'arch', 'combination', 'seed', 'epoch', 'run_path'}
metric_cols = [c for c in plot_df.columns if c not in id_cols]
metric_cols

## Curvas por métrica (una línea por modelo)

Para evitar múltiples líneas por semilla/combinación, se promedia por `(arch, epoch)`.

In [ ]:
n_metrics = len(metric_cols)
ncols = 2
nrows = (n_metrics + ncols - 1) // ncols

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 4 * nrows), squeeze=False)

for idx, metric in enumerate(metric_cols):
    ax = axes[idx // ncols][idx % ncols]
    tmp = (
        plot_df[['arch', 'epoch', metric]]
        .dropna(subset=[metric])
        .groupby(['arch', 'epoch'], as_index=False)[metric]
        .mean()
    )

    sns.lineplot(data=tmp, x='epoch', y=metric, hue='arch', ax=ax, linewidth=2)
    ax.set_title(metric)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(metric)
    ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')

# Oculta ejes sobrantes si el número de métricas es impar
for j in range(n_metrics, nrows * ncols):
    axes[j // ncols][j % ncols].axis('off')

plt.tight_layout()
plt.show()